# Model evaluation on test CSVs

Runs the saved XGBoost model against all `test_*/*.csv` files and reports per-file predictions, probability scores, average inference time, and overall accuracy (labels inferred from filenames containing `fall` vs `walk`).

In [5]:
import sys, time
from pathlib import Path
import pandas as pd
import joblib

# Point to the folder that contains model_utils.py and artifacts
candidates = [
    Path("jupyter-notebook/data-collection/data/ai-model").resolve(),
    Path("data-collection/data/ai-model").resolve(),
]
AI_MODEL_DIR = next((p for p in candidates if (p / "model_utils.py").exists()), None)
if AI_MODEL_DIR is None:
    raise FileNotFoundError("Could not locate model_utils.py. Check your working directory and path candidates.")
sys.path.insert(0, str(AI_MODEL_DIR))

from model_utils import extract_features_signal  # noqa: E402

# Sampling rate used for feature extraction; set to match your device (e.g., 27 for Arduino stream)
FS = 50

model = joblib.load(AI_MODEL_DIR / "best_xgb_model.joblib")
scaler = joblib.load(AI_MODEL_DIR / "scaler.joblib")

# Collect all test CSVs under .../data-collection/data/test_*/
data_candidates = [
    Path("jupyter-notebook/data-collection/data").resolve(),
    Path("data-collection/data").resolve(),
]
base = next((p for p in data_candidates if p.exists()), None)
if base is None:
    raise SystemExit("No data directory found. Adjust data_candidates to your path.")
csv_files = sorted(base.glob("test_*/*.csv"))
if not csv_files:
    # fallback: recursive search
    csv_files = sorted(base.rglob("test_*/*.csv"))
if not csv_files:
    raise SystemExit(f"No CSV files found under {base}")

results = []
labelled = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    required = {"acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"}
    if not required.issubset(df.columns):
        continue

    signal = df[["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]].to_numpy()
    t0 = time.perf_counter()
    feats = extract_features_signal(signal, fs=FS)
    feats_scaled = scaler.transform(feats)
    pred = int(model.predict(feats_scaled)[0])
    proba = model.predict_proba(feats_scaled)[0, 1] if hasattr(model, "predict_proba") else None
    dt = time.perf_counter() - t0
    results.append((csv_path.name, pred, proba, dt))

    name_lower = csv_path.name.lower()
    if "fall" in name_lower:
        labelled.append((1, pred))
    elif "walk" in name_lower:
        labelled.append((0, pred))

if not results:
    raise SystemExit("No CSV files processed.")

avg_ms = sum(r[3] for r in results) / len(results) * 1000
print(f"Processed {len(results)} files; avg inference {avg_ms:.2f} ms (fs={FS} Hz)")
for name, pred, proba, dt in results:
    prob_str = f"{proba:.4f}" if proba is not None else "n/a"
    print(f"{name}: pred={pred}, proba={prob_str}, time={dt*1000:.2f} ms")

if labelled:
    correct = sum(1 for label, p in labelled if label == p)
    acc = correct / len(labelled) * 100
    print(f"\nLabelled files: {len(labelled)} | Accuracy (from filenames): {acc:.2f}%")


Processed 200 files; avg inference 1.05 ms (fs=50 Hz)
test_1_falling_10_124samples_5s.csv: pred=1, proba=0.5355, time=1.89 ms
test_1_falling_11_125samples_5s.csv: pred=1, proba=0.6585, time=1.10 ms
test_1_falling_12_125samples_5s.csv: pred=1, proba=0.7549, time=1.13 ms
test_1_falling_13_124samples_5s.csv: pred=1, proba=0.6628, time=0.78 ms
test_1_falling_14_125samples_5s.csv: pred=1, proba=0.7508, time=0.91 ms
test_1_falling_15_125samples_5s.csv: pred=1, proba=0.7629, time=1.26 ms
test_1_falling_16_119samples_5s.csv: pred=1, proba=0.7501, time=2.08 ms
test_1_falling_17_125samples_5s.csv: pred=1, proba=0.6815, time=1.19 ms
test_1_falling_18_124samples_5s.csv: pred=1, proba=0.7654, time=1.32 ms
test_1_falling_19_124samples_5s.csv: pred=1, proba=0.6565, time=0.88 ms
test_1_falling_1_124samples_5s.csv: pred=0, proba=0.4724, time=0.65 ms
test_1_falling_20_124samples_5s.csv: pred=1, proba=0.5733, time=0.93 ms
test_1_falling_2_125samples_5s.csv: pred=1, proba=0.6701, time=0.84 ms
test_1_falli

In [6]:
import pandas as pd

df = pd.DataFrame(results, columns=["name", "pred", "proba", "dt"])
fall_df = df[df["name"].str.contains("fall", case=False)]
print(f"Fall files: {len(fall_df)}")
print(f"Mean prob: {fall_df['proba'].mean():.4f}")
print(f"Median prob: {fall_df['proba'].median():.4f}")
print(f"Min/Max prob: {fall_df['proba'].min():.4f} / {fall_df['proba'].max():.4f}")
print(f"Pct > 0.5: {(fall_df['proba'] > 0.5).mean()*100:.2f}%")


Fall files: 100
Mean prob: 0.6966
Median prob: 0.7204
Min/Max prob: 0.3566 / 0.7839
Pct > 0.5: 97.00%


In [7]:
import pandas as pd

df = pd.DataFrame(results, columns=["name", "pred", "proba", "dt"])
fall_mask = df["name"].str.contains("fall", case=False)
walk_mask = df["name"].str.contains("walk", case=False)

fall_probs = df.loc[fall_mask, "proba"]
walk_probs = df.loc[walk_mask, "proba"]

print(f"Falls: {fall_probs.size} files")
print(f"  mean={fall_probs.mean():.4f}, median={fall_probs.median():.4f}, min={fall_probs.min():.4f}, max={fall_probs.max():.4f}")
print(f"Walks: {walk_probs.size} files")
print(f"  mean={walk_probs.mean():.4f}, median={walk_probs.median():.4f}, min={walk_probs.min():.4f}, max={walk_probs.max():.4f}")

# Optional: quick accuracy check from filenames
if fall_probs.size + walk_probs.size:
    acc = (labelled.count((1,1)) + labelled.count((0,0))) / len(labelled) * 100
    print(f"Filename-derived accuracy: {acc:.2f}%")


Falls: 100 files
  mean=0.6966, median=0.7204, min=0.3566, max=0.7839
Walks: 100 files
  mean=0.3043, median=0.2808, min=0.2197, max=0.6686
Filename-derived accuracy: 97.50%
